**Code-only release candidate.** Market inputs and executed outputs are deliberately absent pending redistribution clearance. See `DATA_SCHEMA.md` before running market-dependent cells. `python src/teaching_interfaces.py` runs deterministic synthetic checks without market data; it does not reproduce the empirical results.

# Payoff Amplitudes and Tail-Loss Queries

## Goal

Companion to Chapter 2, Sections 2.1 and 2.3. Build the probability and payoff registers for a hypothetical call, then reuse the bounded-payoff construction for a tail indicator and excess loss. Controlled RY angles are twice arcsin(sqrt(scaled payoff)); squaring amplitudes therefore recovers the desired payoff fraction.

## Setup

Run this notebook from top to bottom. The experiment uses local simulation only; no cloud account, trading connection or API key is needed. Market observations stop on **31 August 2026**. The supplied data are quoted-price proxies, not verified total-return series.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Resolve only the package itself; no author-specific absolute path is needed.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "daily_market.csv").exists(), "Authorised daily inputs are required; see DATA_SCHEMA.md. Synthetic checks run without market data via src/teaching_interfaces.py."
sys.path.insert(0, str(ROOT / "src"))
from common import DATA, RESULTS, market


## 1. Reproduce pricing and risk calculations

Historical AAPL volatility calibrates an explicitly hypothetical non-dividend-paying model with S0=K=100, r=3% and T=0.25. Future payoffs are synthetic. MLAE uses genuine small-register amplification powers 0, 1, 2 and 4, with 250 shots each.

In [ ]:
from pricing_risk import run
summary = run()
display(pd.Series({k: summary[k] for k in ["calibration_end", "calibration_n", "sigma", "black_scholes", "mlae_target", "preparation_calls", "mlae_rmse", "direct_rmse"]}, name="Pricing").to_frame())

## 2. Separate grid bias from measurement noise

The continuous analytical price and the discrete 16-bin target are different. The finite-query comparison measures errors around the discrete target. Finer bins can reduce representation bias but require larger preparation circuits.

In [ ]:
display(pd.read_csv(RESULTS / "pricing_discretisation.csv").round(6))
display(Image(filename=str(RESULTS / "pricing_errors.png")))

## 3. Interpret tail quantities

The strict probability beyond a discrete 95% quantile need not equal 5%. Expected shortfall uses q + E[(L-q)+]/0.05, which accounts for the needed part of the atom at q. The quantile is fixed classically in this executed experiment.

In [ ]:
display(pd.Series(summary["risk"], name="Risk output").to_frame())
display(pd.read_csv(RESULTS / "risk_grid.csv").round(6))

## 4. Extend to Asian paths and correlated factors

Two four-bin Gaussian increments generate sixteen two-date paths. An actual five-qubit payoff lookup checks the discrete expectation; independent continuous Monte Carlo has the same two monitoring dates. The 32-date comparator prices a different monitoring contract. Factor scenarios retain all eight eigenvectors and compare dependence assumptions before any loss is encoded.

In [ ]:
from teaching_cases import run as run_extensions
extensions = run_extensions()
display(pd.Series(extensions["asian"], name="Two-date Asian example").to_frame())
display(pd.read_csv(RESULTS / "factor_scenarios.csv").round(6))

## Interpretation and Next Steps

Read the grid and measurement errors separately. More shots cannot eliminate coarse path-grid bias. The Asian circuit loads an enumerated payoff lookup; it does not implement scalable reversible path generation. In the factor example, a negative leading-factor shift is a chosen stress rather than a forecast. Try varying one representation setting while keeping the financial contract fixed.